# ohsome quality API (OQAPI) 

OQAPI provides quality estimations of OpenStreetMap (OSM) data.
OQAPI calls those estimations Indicators.
Indicators are computed for a specific area (`bpolys`) and a certain set of aggregated OSM features (`topic`).

Two intrinsic quality estimation OQAPI offers are the Mapping Saturation indicator and the Currentness indicator.

## API Request Examples using Python

Below Python is used to make requests to the ohsome quality API.

## Authentication

Requests to the ohsome quality API require an API key. Get a free key at
[account.heigit.org](https://account.heigit.org/) and set it as the
`api_key` variable.

In [ ]:
api_key = "YOUR_API_KEY"

## Topics

Every indicator request needs a `topic` key identifying the OSM feature set to
analyze. Look at the predefined topics [topics page](https://giscience.github.io/oqapi-jupyter-book/en/topics)
in the oqapi-jupyter-book, which predates the OQAPI v2 topic rename/cleanup and
still lists dead keys such as `building-count`/`building-area` and
`industrial-landuse-count`/`industrial-landuse-area` (now unified into single
topics) and is missing newer ones such as `hospitals`.

| Key | Name | Description |
| --- | --- | --- |
| `bridges_all_ways` | Bridges (all ways) | All linear features tagged as a bridge, including footpaths |
| `bridges_cars` | Bridges (cars) | Bridges usable by vehicles |
| `buildings` | Buildings | All `building=*` polygons (excluding `building=no`) |
| `bus-stops` | Bus Stops | Count of `highway=bus_stop` points |
| `clinics` | Clinics | Count of clinic amenities/healthcare features |
| `custom-topic` | Custom Topic | User-defined ohsome filter (requires `topicFilter`, `topicTitle` and `measure`) |
| `cycleway` | Cycleway | Exclusive cycleways and cycleways alongside streets |
| `doctors` | Doctors | Count of doctors' offices |
| `fire-stations` | Fire Stations | Count of fire stations |
| `fitness-centres` | Fitness Centres | Count of gyms/fitness/sports centres |
| `footpath` | Footpath | Pedestrian paths, shared paths, sidewalks, foot-allowed roads |
| `footway` | Footway | Dedicated pedestrian ways, sidewalks, crossings |
| `forests` | Forests | Area tagged `landuse=forest` |
| `hospitals` | Hospitals | Count of hospitals |
| `industrial-landuse` | Industrial Landuse | Area tagged `landuse=industrial` |
| `kindergarten` | Kindergartens | Count of kindergartens |
| `land-cover` | Land Use and Land Cover | Broad land-use/land-cover polygon features |
| `marketplaces` | Marketplaces | Count of marketplaces |
| `poi` | POI | Broad points-of-interest filter (amenities, shops, tourism, etc.) |
| `parks` | Parks | Count of `leisure=park` |
| `power_lines` | Power Lines | `power=line`/`minor_line` linear features |
| `power_substations` | Power Substation | `power=substation` facilities |
| `public-transport-stops` | Public Transport Stops | `public_transport=platform` features |
| `railways` | Railways | Rail/subway/tram/light-rail/monorail/funicular/narrow-gauge line length |
| `roads-all-highways` | Roads (all highways) | All `highway=*` linear features, including links |
| `roads` | Roads (cars) | Car-usable road classes (motorway…unclassified, plus links) |
| `schools` | Schools | Count of schools |
| `sports-pitch` | Sports Pitches | Count of `leisure=pitch` |
| `subway-stations` | Subway Stations | Count of `station=subway` |
| `supermarkets` | Supermarkets | Count of supermarkets/convenience stores |
| `tram-stops` | Tram Stops | Count of `railway=tram_stop` |

Every topic works with Mapping Saturation, Currentness, Attribute Completeness
and User Activity. A few topics additionally unlock a reference-dataset-backed
indicator: `buildings` → Building Comparison, `roads-all-highways` → Road
Comparison, `roads` → Roads Thematic Accuracy, `land-cover` → Land Cover
Completeness and Land Cover Thematic Accuracy.

### Mapping Saturation Indicator

The Mapping Saturation indicator calculate the saturation of mapping activity within the last 3 years.
It is based on the premise that each aggregation of features (e.g. length of roads or count of buildings) has a maximum. After increased mapping activity saturation is reached near this maximum.
The Mapping Saturation indicator works well with following topics:

- Buildings (`buildings`)
- Roads (`roads-all-highways`)
- Points of Interest (`poi`)

See the [Mapping Saturation page](https://giscience.github.io/oqapi-jupyter-book/en/mapping-saturation) in the oqapi-jupyter-book for more details.

In [ ]:
import json

import plotly.graph_objects as go
import requests
from IPython.display import display as ipy_display

base_url = "https://api.heigit.org/ohsome-quality-api/v2"
endpoint = "/indicators"
indicator = "/mapping-saturation"
url = base_url + endpoint + indicator

with open("germany-heidelberg.geojson", "r") as file:
    bpolys = json.load(file)

headers = {"accept": "application/json", "authorization": api_key}

parameters = {
    "topic": "buildings",
    "bpolys": bpolys,
}

response = requests.post(url, headers=headers, json=parameters, timeout=60)
response.raise_for_status()
results = response.json()["result"]

# display plotly figure per feature
for result in results:
    figure = go.Figure(result["result"]["figure"])
    ipy_display(figure)

    ## convert figure to SVG
    # svg = pio.to_image(figure, format='svg')
    # display(SVG(svg))

#### Custom Topic Example

Instead of one of the predefined topic keys, you can define your own topic
inline by setting `topic` to `custom-topic` and providing a `topicFilter` (an
[ohsome API filter](https://giscience.github.io/oqapi-jupyter-book/en/custom-topics)
expression), a `topicTitle`, and a `measure` (`count`, `length` or `area`).
This works with any indicator, not just Mapping Saturation.

In [ ]:
import json

import plotly.graph_objects as go
import requests
from IPython.display import display as ipy_display

base_url = "https://api.heigit.org/ohsome-quality-api/v2"
endpoint = "/indicators"
indicator = "/mapping-saturation"
url = base_url + endpoint + indicator

with open("germany-heidelberg.geojson", "r") as file:
    bpolys = json.load(file)

headers = {"accept": "application/json", "authorization": api_key}

parameters = {
    "topic": "custom-topic",
    "topicTitle": "Trees",
    "topicFilter": "natural=tree and geometry:point",
    "measure": "count",
    "bpolys": bpolys,
}

response = requests.post(url, headers=headers, json=parameters, timeout=60)
response.raise_for_status()

results = response.json()["result"]

# display plotly figure per feature
for result in results:
    figure = go.Figure(result["result"]["figure"])
    ipy_display(figure)

### Curretness Indicator

The Currentness indicator works well with following topics:

- Buildings (`buildings`)
- Roads (`roads-all-highways`)
- Points of Interest (`poi`)

See the [Currentness page](https://giscience.github.io/oqapi-jupyter-book/en/currentness) in the oqapi-jupyter-book for more details.

In [ ]:
import json

import plotly.graph_objects as go
import requests
from IPython.display import display as ipy_display

base_url = "https://api.heigit.org/ohsome-quality-api/v2"
endpoint = "/indicators"
indicator = "/currentness"
url = base_url + endpoint + indicator

with open("germany-heidelberg.geojson", "r") as file:
    bpolys = json.load(file)

headers = {"accept": "application/json", "authorization": api_key}

parameters = {
    "topic": "buildings",
    "bpolys": bpolys,
}

response = requests.post(url, headers=headers, json=parameters, timeout=60)
response.raise_for_status()

results = response.json()["result"]

# display plotly figure per feature
for result in results:
    figure = go.Figure(result["result"]["figure"])
    ipy_display(figure)

### Attribute Completeness Indicator

The Attribute Completeness indicator measures how many features matching a
topic also carry an additional expected attribute — e.g. how many buildings
have height information, or how many roads have a `surface` tag. It is a
semantic completeness check: the topic finds the features that should exist,
the attribute check tests how well-described they are.

See the [Attribute Completeness page](https://giscience.github.io/oqapi-jupyter-book/en/attribute-completeness) in the oqapi-jupyter-book for more details.

In [ ]:
import json

import plotly.graph_objects as go
import requests
from IPython.display import display as ipy_display

base_url = "https://api.heigit.org/ohsome-quality-api/v2"
endpoint = "/indicators"
indicator = "/attribute-completeness"
url = base_url + endpoint + indicator

with open("germany-heidelberg.geojson", "r") as file:
    bpolys = json.load(file)

headers = {"accept": "application/json", "authorization": api_key}

parameters = {
    "topic": "buildings",
    "attributes": ["height"],
    "bpolys": bpolys,
}

response = requests.post(url, headers=headers, json=parameters, timeout=60)
response.raise_for_status()

results = response.json()["result"]

# display plotly figure per feature
for result in results:
    figure = go.Figure(result["result"]["figure"])
    ipy_display(figure)

#### Custom Attribute Example

Instead of picking from OQAPI's predefined attribute keys, you can check completeness for any tag by providing your own
`attributeFilter` (an ohsome API filter expression) together with an
`attributeTitle`, instead of `attributes`.

In [ ]:
import json

import plotly.graph_objects as go
import requests
from IPython.display import display as ipy_display

base_url = "https://api.heigit.org/ohsome-quality-api/v2"
endpoint = "/indicators"
indicator = "/attribute-completeness"
url = base_url + endpoint + indicator

with open("germany-heidelberg.geojson", "r") as file:
    bpolys = json.load(file)

headers = {"accept": "application/json", "authorization": api_key}

parameters = {
    "topic": "buildings",
    "attributeTitle": "Roof Shape",
    "attributeFilter": "roof:shape=*",
    "bpolys": bpolys,
}

response = requests.post(url, headers=headers, json=parameters, timeout=60)
response.raise_for_status()

results = response.json()["result"]

# display plotly figure per feature
for result in results:
    figure = go.Figure(result["result"]["figure"])
    ipy_display(figure)

### Building Comparison Indicator

The Building Comparison indicator estimates completeness by comparing the
total OSM building area within an area of interest against reference building
datasets — EUBUCCO (administrative building footprints for Europe) and
Microsoft Building Footprints (satellite-derived, global). The result is the
ratio of OSM building area to reference building area, clipped to whichever
part of the area of interest the reference dataset actually covers.

See the [Building Comparison page](https://giscience.github.io/oqapi-jupyter-book/en/building-comparison) in the oqapi-jupyter-book for more details.

In [ ]:
import json

import plotly.graph_objects as go
import requests
from IPython.display import display as ipy_display

base_url = "https://api.heigit.org/ohsome-quality-api/v2"
endpoint = "/indicators"
indicator = "/building-comparison"
url = base_url + endpoint + indicator

with open("germany-heidelberg.geojson", "r") as file:
    bpolys = json.load(file)

headers = {"accept": "application/json", "authorization": api_key}

parameters = {
    "topic": "buildings",
    "bpolys": bpolys,
}

response = requests.post(url, headers=headers, json=parameters, timeout=60)
response.raise_for_status()

results = response.json()["result"]

# display plotly figure per feature
for result in results:
    figure = go.Figure(result["result"]["figure"])
    ipy_display(figure)

### Road Comparison Indicator

The Road Comparison indicator estimates completeness by comparing the total
length of OSM roads within an area of interest against a reference road
dataset derived from satellite imagery (Microsoft Roads/RoadDetections). The
result is the ratio of OSM road length to reference road length, clipped to
whichever part of the area of interest the reference dataset covers. Only the
`roads-all-highways` topic is supported.

See the [Road Comparison page](https://giscience.github.io/oqapi-jupyter-book/en/road-comparison) in the oqapi-jupyter-book for more details.

In [ ]:
import json

import plotly.graph_objects as go
import requests
from IPython.display import display as ipy_display

base_url = "https://api.heigit.org/ohsome-quality-api/v2"
endpoint = "/indicators"
indicator = "/road-comparison"
url = base_url + endpoint + indicator

with open("germany-heidelberg.geojson", "r") as file:
    bpolys = json.load(file)

headers = {"accept": "application/json", "authorization": api_key}

parameters = {
    "topic": "roads-all-highways",
    "bpolys": bpolys,
}

response = requests.post(url, headers=headers, json=parameters, timeout=60)
response.raise_for_status()

results = response.json()["result"]

# display plotly figure per feature
for result in results:
    figure = go.Figure(result["result"]["figure"])
    ipy_display(figure)

### Land Cover Completeness Indicator

The Land Cover Completeness indicator measures what share of an area of
interest's surface is covered by any OSM land-cover/land-use feature. It is a
spatial coverage check — the ratio of mapped area to the area of interest's
total area — rather than a check of whether the mapped features are correctly
classified. Only the `land-cover` topic is supported.

See the [Land Cover Completeness page](https://giscience.github.io/oqapi-jupyter-book/en/land-cover-completeness) in the oqapi-jupyter-book for more details.

In [ ]:
import json

import plotly.graph_objects as go
import requests
from IPython.display import display as ipy_display

base_url = "https://api.heigit.org/ohsome-quality-api/v2"
endpoint = "/indicators"
indicator = "/land-cover-completeness"
url = base_url + endpoint + indicator

with open("germany-heidelberg.geojson", "r") as file:
    bpolys = json.load(file)

headers = {"accept": "application/json", "authorization": api_key}

parameters = {
    "topic": "land-cover",
    "bpolys": bpolys,
}

response = requests.post(url, headers=headers, json=parameters, timeout=60)
response.raise_for_status()

results = response.json()["result"]

# display plotly figure per feature
for result in results:
    figure = go.Figure(result["result"]["figure"])
    ipy_display(figure)

### Land Cover Thematic Accuracy Indicator

The Land Cover Thematic Accuracy indicator checks whether OSM land-cover
classifications are correct, not just present. It intersects OSM land-cover
polygons with the CORINE Land Cover (CLC) reference dataset and builds a
confusion matrix per land-cover class, reporting classification metrics
(precision, recall, F1) at both a coarse and a detailed CLC class level. Only
the `land-cover` topic is supported. Currently only available for Germany.

See the [Land Cover Thematic Accuracy page](https://giscience.github.io/oqapi-jupyter-book/en/land-cover-thematic-accuracy) in the oqapi-jupyter-book for more details.

In [ ]:
import json

import plotly.graph_objects as go
import requests
from IPython.display import display as ipy_display

base_url = "https://api.heigit.org/ohsome-quality-api/v2"
endpoint = "/indicators"
indicator = "/land-cover-thematic-accuracy"
url = base_url + endpoint + indicator

with open("germany-heidelberg.geojson", "r") as file:
    bpolys = json.load(file)

headers = {"accept": "application/json", "authorization": api_key}

parameters = {
    "topic": "land-cover",
    "bpolys": bpolys,
}

response = requests.post(url, headers=headers, json=parameters, timeout=60)
response.raise_for_status()

results = response.json()["result"]

# display plotly figure per feature
for result in results:
    figure = go.Figure(result["result"]["figure"])
    ipy_display(figure)

### Roads Thematic Accuracy Indicator

The Roads Thematic Accuracy indicator compares OSM road attributes (`surface`,
`oneway`, `lanes`, `name`, `width`) against Germany's official reference road
network, the Digitales Landschaftsmodell (DLM) from the Bundesamt für
Kartographie und Geodäsie (BKG). It matches OSM and DLM road geometries and
reports agreement/disagreement statistics for the selected attribute. This
indicator only covers Germany, which is why the example below reuses
`germany-heidelberg.geojson`. Only the `roads` topic is supported.

See the [Road Thematic Accuracy page](https://giscience.github.io/oqapi-jupyter-book/en/road-thematic-accuracy) in the oqapi-jupyter-book for more details.

In [ ]:
import json

import plotly.graph_objects as go
import requests
from IPython.display import display as ipy_display

base_url = "https://api.heigit.org/ohsome-quality-api/v2"
endpoint = "/indicators"
indicator = "/roads-thematic-accuracy"
url = base_url + endpoint + indicator

with open("germany-heidelberg.geojson", "r") as file:
    bpolys = json.load(file)

headers = {"accept": "application/json", "authorization": api_key}

parameters = {
    "topic": "roads",
    "attribute": "surface",  # empty string for all attributes
    "bpolys": bpolys,
}

response = requests.post(url, headers=headers, json=parameters, timeout=60)
response.raise_for_status()

results = response.json()["result"]

# display plotly figure per feature
for result in results:
    figure = go.Figure(result["result"]["figure"])
    ipy_display(figure)

### User Activity Indicator

The User Activity indicator reports the number of unique OSM contributors
active per month for a topic within an area of interest. Unlike the other
indicators it has no quality dimension and produces no pass/fail label — it's
contextual/diagnostic information about how active the mapping community is,
useful for interpreting the other indicators (e.g. low currentness alongside
low user activity suggests a lack of contributors, not a data problem).

See the [User Activity page](https://giscience.github.io/oqapi-jupyter-book/en/user-activity) in the oqapi-jupyter-book for more details.

In [ ]:
import json

import plotly.graph_objects as go
import requests
from IPython.display import display as ipy_display

base_url = "https://api.heigit.org/ohsome-quality-api/v2"
endpoint = "/indicators"
indicator = "/user-activity"
url = base_url + endpoint + indicator

with open("germany-heidelberg.geojson", "r") as file:
    bpolys = json.load(file)

headers = {"accept": "application/json", "authorization": api_key}

parameters = {
    "topic": "buildings",
    "bpolys": bpolys,
}

response = requests.post(url, headers=headers, json=parameters, timeout=60)
response.raise_for_status()

results = response.json()["result"]

# display plotly figure per feature
for result in results:
    figure = go.Figure(result["result"]["figure"])
    ipy_display(figure)